# Finding lensed counterparts for each source

This notebook detects likely lensed counterparts using the Lyman alpha line profiles of each spectrum. Additionally, cleaning of possible [OII] doublet contamination is performed.

In [ ]:
import numpy as np
from astropy.io import fits
import astropy.table as aptb
import matplotlib.pyplot as plt
import copy
import shutil

from tangelo import fitting as tgfit
from tangelo import constants as tgconst
from tangelo import catalogue_operations as tgcat
from tangelo import lya_fitting as lyafit
from tangelo import io as tgio

In [ ]:
# Get c from constants
c = tgconst.c

In [ ]:
def rename_spec_file(old_iden, new_iden, cluster, spec_source, spec_type):
    """
    Rename the spectrum file corresponding to the old identifier to have the new identifier in its name. 
    This is necessary because the spectra files are named according to the identifiers in the R21 catalogues,
    which we are changing for some objects to avoid duplicates.

    Parameters
    ----------
    old_iden : int
        The old identifier of the object, which is used in the original spectra file name.
    new_iden : int
        The new identifier of the object, which we want to have in the spectra file name.
    cluster : str
        The name of the cluster (e.g. 'MACS0416S') which is part of the spectra file path.
    spec_source : str
        The source of the spectra files, either 'R21' or 'APER', which determines the directory where the spectra are stored.
    spec_type : str
        The type of spectra, e.g. '1fwhm', 'noweight_skysub', which also affects the file name pattern.

    Returns
    -------
    None
    """
    spec_get_meth = tgio.get_r21_spectra_dir if spec_source == 'R21' else tgio.get_aper_spectra_dir
    spec_dir = spec_get_meth(cluster)
    fname = f"{'spec_' if spec_source == 'R21' else ''}{old_iden}_{spec_type}.fits"
    spec_path = spec_dir / fname
    if spec_path.exists():
        new_fname = f"{'spec_' if spec_source == 'R21' else ''}{new_iden}_{spec_type}.fits"
        new_spec_path = spec_dir / new_fname
        shutil.copy(spec_path, new_spec_path)
        print(f"Copied {spec_path} to {new_spec_path}")
    else:
        print(f"Warning: Spectrum file {spec_path} does not exist and cannot be renamed.")

In [ ]:
SPEC_SOURCE    = 'R21' # 'APER' or 'R21'
SPEC_TYPE      = 'weight_skysub' # '1fwhm_opt','2fwhm', etc.
COUNTER_SOURCE = 'R21' # 'R21' uses lensed counterparts from R21, 'APER' looks for similar Lyman alpha profiles to identify
                        # lensed counterparts

# Where are spectra stored
spec_dir = tgio.get_r21_spectra_dir if SPEC_SOURCE == 'R21' else tgio.get_aper_spectra_dir
filepattern = lambda iden: f"{'spec_' if SPEC_SOURCE == 'R21' else ''}{iden}_{SPEC_TYPE}.fits"

# Load the megatab generated from the fitting results of chosen source
megatab = aptb.Table(fits.open(f'./megatables/fit_catalog_qc_{SPEC_TYPE}.fits')[1].data)

In [ ]:
# Dictionary in which we load the R21 catalogues that will be used to check lensed counterparts
cluslist = np.unique(megatab['CLUSTER'])  # First get list of all clusters
lenstables = tgcat.make_r21_catalogue_dict(cluslist)

# Make a column to tell us where they came from
for clus, tab in lenstables.items():
    tab['CLUSTER'] = np.array([clus for _ in tab])
    
# To deal with the case of MACS0416, where there are two pointings that use the same idens for different sources,
# we must find any repeated idens and change them to make them unique.
for row in lenstables['MACS0416NE']:
    same_rows = (lenstables['MACS0416S']['iden'] == row['iden']) * (lenstables['MACS0416S']['idfrom'] == row['idfrom'])
    iden = f"{row['idfrom'][0].replace('E','X')}{row['iden']}" # Get the full identifier
    if same_rows.sum() > 0:
        print(f"{iden} exists in both S and NE pointings")
        idx = np.where(same_rows)[0][0]
        # Change the entry to have an S at the end in the S table
        new_iden = lenstables['MACS0416S']['iden'][idx] * 100 + 55
        new_iden_full = f"{row['idfrom'][0].replace('E','X')}{new_iden}"
        lenstables['MACS0416S']['iden'][idx] = new_iden
        # Find the entry from the megatab with the same identifier and cluster
        mega_idx = np.where((megatab['CLUSTER'] == 'MACS0416S') * 
                            (megatab['iden'] == iden))[0]
        if np.size(mega_idx) > 0:
            # Change the identifier in the megatab to match the new one in the S table
            megatab['iden'][mega_idx[0]] = new_iden_full
            print(f"Replaced {iden} with {megatab['iden'][mega_idx[0]]} in megatab")
            print(megatab['iden'][mega_idx[0]])
        # Finally, make copies of the spectra with the new names
        rename_spec_file(iden, new_iden_full, 'MACS0416S', SPEC_SOURCE, SPEC_TYPE)

# Stack the tabs from MACS0416
lenstables['MACS0416'] = aptb.vstack([lenstables['MACS0416NE'], lenstables['MACS0416S']])

del lenstables['MACS0416NE']
del lenstables['MACS0416S']

In [ ]:
# Split the megatab into separate tables for each cluster
clustertabs = {
    clus: megatab[megatab['CLUSTER'] == clus] for clus in cluslist
}

# Put both MACS0416 tables together to identify all lensed counterparts
clustertabs['MACS0416'] = aptb.vstack([clustertabs['MACS0416NE'], clustertabs['MACS0416S']])

# Remove the separate tables
del clustertabs['MACS0416NE']
del clustertabs['MACS0416S']

# Verify that there are no duplicated idens in the combined MACS0416 table
unique_idens, counts = np.unique(clustertabs['MACS0416']['iden'], return_counts=True)
if len(unique_idens) != len(clustertabs['MACS0416']):
    duplicate_idens = unique_idens[counts > 1]
    print(f"Duplicate idens found:\n {duplicate_idens}")
    raise ValueError("There are duplicated idens in the combined MACS0416 table!")

In [ ]:
def sort_table(tab):
    """
    Sort a lensing catalogue by Lyalpha SNR, putting contaminated sources at the end, and 
    within each group putting PRIOR sources first, then EXTERN/MUSELET. So there should be 
    4 groups in total, each sorted by Lyalpha SNR.

    Parameters
    ----------
    tab: astropy.table.Table
        Astropy table of lensing catalogue entries
    
    Returns
    -------
    tab_sorted: astropy.table.Table
        A new sorted Astropy table
    """
    # First make numerical columns corresponding to the degree of contamination and idfrom
    cnumber = np.array([int(np.array([r[cn] == 'c' for cn in tab.colnames]).sum()) for r in tab])
    tab['CONTAM'] = cnumber
    from_numbers = {'PRIOR': 1, 'EXTERN':0, 'MUSELET': 0}
    tab['PRIOR'] = np.array([from_numbers[r['idfrom']] for r in tab]).astype(bool)
    
    # Now split the table into those that are contaminated and those that arent
    cont_free  = tab['CONTAM'] == 0
    cont_split = [tab[cont_free], tab[~cont_free]]
    # Now split each of these into PRIOR and non-PRIOR
    final_split = []
    for subt in cont_split:
        prior = subt['PRIOR']
        final_split.append([subt[prior], subt[~prior]])
    
    # Now sort each of these by Lyalpha SNR
    for i in range(2):
        for j in range(2):
            final_split[i][j].sort('SNRR', reverse=True)
    
    # Finally stack them all back together
    tab_sorted = aptb.vstack([final_split[0][0], final_split[0][1], final_split[1][0], final_split[1][1]])
    tab_sorted.remove_columns(['PRIOR'])

    return tab_sorted

In [ ]:
sortabs = []

# Now iterate over CLUSTER tabs, rather than just tabs
for clus, tab in clustertabs.items():
    
    print(f"Processing {clus}...")
    
    # Separate out contaminated sources to the bottom so that they can't be the lead member of a group
    sortab = sort_table(tab)
    counters = np.array([0 for i in range(len(sortab))]).astype(str)
    skiplist = []
    
    # dictionary that will hold information about the strength of the last match to each object:
    matchsth = {}
    for i, row in enumerate(sortab[:-1]):
        if i in skiplist:
            continue
        for j, othrow in enumerate(sortab[i+1:]):
            match_result = tgcat.is_counterpart(row, othrow, method = COUNTER_SOURCE, lenstables = lenstables)
            if match_result[0]:
                print(f"Found counterpart for {clus} {row['iden']} ({othrow['iden']})")
                if i+j+1 in skiplist: #if the match has been matched before:
                    if match_result[1] > matchsth[i+j+1]: #and that match was a better match than this one:
                        print(f"Previous match was better, skipping...")
                        continue #then skip this match and go to the next one; otherwise, this match will now belong to the new source
                    else:
                        print(f"Overwriting old match...")
                counters[i]     = row['iden']
                counters[i+j+1] = row['iden']
                skiplist.append(i+j+1)
                matchsth[i+j+1] = match_result[1] #add match strength info (n_sig difference in LPEAK) to the dictionary
                
    # See how many times each counter ID appears
    countercounts = np.unique(counters, return_index=True, return_counts=True)
    
    # Make an array of all the indices where the counter only appears once and then set them all to zero (that means no counter)
    blist = countercounts[1][countercounts[2] == 1]
    counters[blist] = '0'
    
    # Insert into table
    sortab['COUNTER'] = counters
    sortabs.append(sortab)

megatab_reform = aptb.vstack(sortabs)

In [ ]:
# Save the megatab with lensed counterparts
megatab_reform.write(f'megatables/fit_catalog_qc_cpts_{SPEC_TYPE}.fits', overwrite=True)

In [ ]:
def delete_bad_counters(table, bad_table):
    """
    Delete rows from the megatab corresponding to contaminated counterparts that we want to discard. This is done in place.

    Parameters
    ----------
    table: astropy.table.Table
        The megatab from which we want to delete rows
    bad_table: astropy.table.Table
        A table containing the rows that we want to delete from the megatab, which should be a subtable of the megatab itself, 
        so that the column names and formats are the same.
        
    Returns
    -------
    None
    """
    # Create a mask of rows to KEEP
    mask = np.ones(len(table), dtype=bool)
    
    # Loop over rows in the bad table and mark matching rows for removal
    for bad_row in bad_table:
        # Create a condition for each column
        condition = np.ones(len(table), dtype=bool)
        matchcols = ['iden', 'idfrom', 'MU', 'LPEAKR']
        for colname in matchcols:
            condition &= (table[colname] == bad_row[colname])
        
        # Mark matching rows as False (to be removed)
        mask &= ~condition
    
    print(f"Removing {(~mask).sum()} bad counterparts from the megatab.")
    
    # Remove the rows in place using the mask
    table.remove_rows(np.where(~mask)[0])

def remove_contaminants(subtab, source):
    """
    Remove contaminated sources from a subtable of the megatab corresponding to a single group of counterparts.

    Parameters
    ----------
    subtab: astropy.table.Table
        A subtable of the megatab containing only the sources in a single group of counterparts
    source: str
        The identifier of the lead source in the group, which we want to make sure is not contaminated

    Returns
    -------
    subtab_clean: astropy.table.Table
        A new subtable with contaminated sources removed, except the lead source if it is contaminated,
        in which case an error is raised because we cannot discard the lead source of a group.
    """


    # Critical lines to be checked for contamination
    crit_lines = ['NV1238', 'SiII1260', 'SiIV1394', 'CIV1548', 'HeII1640', 'OIII1660', 'SiIII1883', 'CIII1909']
    
    # index of the leader
    lead_idx = np.where(subtab['iden'] == source)[0][0]

    # Make list of flag columns for the critical lines
    flagcols = []
    for cn in subtab.colnames:
        if 'FLAG_' in cn and cn.split('_')[-1] in crit_lines:
            flagcols.append(cn)
    
    # Which sources are contaminated?
    contams = np.array([subtab[col] == 'c' for col in flagcols]) # This gets a 2D array with each column representing a Y/N for flag
    contams = np.sum(contams, axis=0).astype(bool) # This sums across the columns, effectively asking "which sources are flagged?"
    if contams.sum() == len(subtab):
        print(f"Warning! all counterparts are contaminated. Keeping the main counterpart along with flags.")
        contams[lead_idx] = False # if all are contaminated, keep the main counterpart
    # Make sure that the leader isn't contaminated -- if it is, this will cause problems later on.
    if contams[lead_idx]:
        raise ValueError(f"CANNOT DISCARD THE TITULAR MEMBER OF A GROUP!")
    if contams.sum():
        print(f"Rejecting {subtab['iden'][contams].data} because of contamination.")
        # Delete the corresponding rows from megatab_reform
        delete_bad_counters(megatab_reform, subtab[contams])
    
    return subtab[~contams]

def get_original_pointing(iden, cluster):
    """
    Get the original pointing (S or NE) of a source in MACS0416, which is necessary to find the correct spectra file.

    Parameters
    ----------
    iden: int
        The identifier of the source, which is not unique across the two pointings of MACS0416, so we need to check both tables to find it.
    cluster: str
        The name of the cluster, which should be 'MACS0416' for this function to work properly.

    Returns
    -------
    pointing: str
        The original pointing of the source, either 'S' or 'NE'.
    """
    if cluster != 'MACS0416':
        return cluster # If it's not MACS0416, we can just return the cluster name as the pointing
    
    clustertab_row = clustertabs[cluster][clustertabs[cluster]['iden'] == iden]

    if len(clustertab_row) == 0:
        raise ValueError(f"Identifier {iden} not found in cluster {cluster} table!")
    elif len(clustertab_row) > 1:
        raise ValueError(f"Identifier {iden} found multiple times in cluster {cluster} table!")
    else:
        return clustertab_row['CLUSTER'][0]

In [ ]:
# Stacking spectra of all multiply-lensed images and perform fitting of Lya and any other lines based on the highest SNR lensed image.
# for the purposes of what we are about to do, need to combine all MACS0416 into one cluster. Note: this is legacy code that
# should be refactored when time permits as it is hard to read, though it works as intended.

# As we are going to add a character to the identifiers of some sources, we need to increase the max length of the iden column
# by changing the dtype to U{max_len+1} where max_len is the current maximum length of the iden entries
max_len = np.max([len(str(s)) for s in megatab_reform['iden']]) # Find the max length of the current iden entries
new_max_len = max_len + 1 # We will add one character to the iden
megatab_reform['iden'] = megatab_reform['iden'].astype(f'U{new_max_len}')

# First, we copy the megatab because we are going to make changes to it
megatab_reform_cp = copy.deepcopy(megatab_reform)
# We then merge MACS0416 to a singe cluster
megatab_reform_cp['CLUSTER'][np.logical_or(megatab_reform['CLUSTER'] == 'MACS0416S', 
                                           megatab_reform['CLUSTER'] == 'MACS0416NE')] = 'MACS0416'

# Loop over clusters
for clus in np.unique(megatab_reform_cp['CLUSTER']):

    print("\n" + "=" * 80)
    print(f"Processing cluster {clus}...")

    # Get the subtab for the cluster
    tab = megatab_reform_cp[megatab_reform_cp['CLUSTER'] == clus]
    
    # Loop over hub names (i.e. the titular source of each group)
    for titular_source in np.unique(tab['COUNTER']):
        if str(titular_source) == '0':
            continue # zero just means no counterparts, so we skip this
        
        print(f"\nAggregating counterimages of source {titular_source} from cluster {clus}...")

        stacked_iden = f"S{titular_source}" # The identifier for the stacked spectrum will be the same as the titular source
                                            # but with an S in front to indicate it's a stack
        
        # Get a mini-tab for the counterpart group (spokes)
        subtab = tab[tab['COUNTER'] == titular_source]
        
        # Check for contaminated lines -- if critical lines are contaminated, exclude the source from the stack
        subtab = remove_contaminants(subtab, titular_source)

        # Print the table
        print(f"Counterpart group:\n")
        print(subtab['iden','CLUSTER', 'SNRB', 'SNRR', 'MU', 'z'])
        
        # Reject any alleged counterparts that have low SNR. We can skip this if using R21 counterparts
        if COUNTER_SOURCE != 'R21':
            if subtab['SNRR'][0] > 10.0 and (subtab['SNRR'] < 5.0).sum() > 0:
                print(f"Rejecting {(subtab['SNRR'] < 5.0).sum()} of {len(subtab)} counterparts due to relatively poor Lya SNR.")
                # Delete from the megatab as otherwise we may double-count (also, less a 5 sig is not fit for
                # analysis anyway).
                delete_bad_counters(megatab_reform, subtab[subtab['SNRR'] < 5.0])
                subtab = subtab[subtab['SNRR'] > 5.0]
                
        # If subtab now only has one row left, no coaddition needs to be done, so we can just skip,
        if len(subtab) == 1:
            print(f"After cleaning, only one counterpart remains! Skipping {clus} source {titular_source}.")
            continue
        
        # Recover the original pointing of each member source (necessary to find spectra files)
        original_clusters = {
            row['iden'] : get_original_pointing(row['iden'], row['CLUSTER']) for row in subtab
        }

        # Original cluster of the titular source
        titular_cluster = original_clusters[titular_source]
        
        # Load up the spectra
        specs   = {
            row['iden']: tgio.load_spec(original_clusters[row['iden']], row['iden'], row['idfrom'], 
                                        spec_type=SPEC_TYPE, spec_source=SPEC_SOURCE) 
            for row in subtab
        }
        
        # Weight the spectra by inverse median variance to avoid adding noise
        weights = {s: 1. / np.nanmedian(specs[s]['spec_err'] ** 2.) for s in specs.keys()}
        weightsum = np.sum(np.array([w for w in weights.values()]))
        # Normalize the weights
        for k in weights.keys():
            weights[k] /= weightsum
        
        # Stacking
        avgspec = np.nansum(np.array([specs[s]['spec'] * weights[s] for s in specs.keys()]), axis=0)
        avgerr = np.sqrt(np.nansum(np.array([(specs[s]['spec_err'] * weights[s]) ** 2. for s in specs.keys()]), axis=0))
                   
        # Calculate weighted equivalent magnification
        avgmu = np.nansum(subtab['MU'].data * np.array(list(weights.values())))
        avgmu_err = np.sqrt(np.nansum((subtab['MU_ERR'].data * np.array(list(weights.values()))) ** 2.))
        # Store the average magnification and error in the avgmags dictionary for later use

        # Update megatab with new information
        titular_index = np.where(np.logical_and(megatab_reform['CLUSTER'] == titular_cluster, 
                                                megatab_reform['iden'] == titular_source))[0][0]
        megatab_reform['iden'][titular_index] = stacked_iden # Update the identifier to be the new one for the stack
        megatab_reform['MU'][titular_index] = avgmu # Update the magnification to be the average one for the stack
        megatab_reform['MU_ERR'][titular_index] = avgmu_err # Update the magnification error to be the average one for the stack
        # Push flags to updated megatab
        for colname in subtab.colnames:
            if 'FLAG_' in colname and 'c' in subtab[colname].data:
                megatab_reform[colname][titular_index] = 'c' # If any of the counterparts are contaminated in a critical line, flag the stack
        
        # Plotting the results
        wln = np.array([s['wave'] for s in specs.values()])[0]
        lya = subtab['LPEAKR'][0]
        lyarange = np.logical_and(lya - 50 < wln, wln < lya + 50)
        fig, ax = plt.subplots(1,2, figsize=(20,5), facecolor='white')
        ax[0].step(wln, avgspec, where='mid')
        ax[0].axvline(lya, color='maroon', linestyle='--')
        ax[1].plot(wln[lyarange], avgspec[lyarange], drawstyle='steps-mid', color='black', alpha=0.75) #plot the stack
        ax[1].fill_between(wln[lyarange], avgspec[lyarange] - avgerr[lyarange], avgspec[lyarange] + avgerr[lyarange],
                       color='black', step='mid', alpha = 0.25)
        for iden, spec in specs.items():
            ax[1].plot(spec['wave'][lyarange], spec['spec'][lyarange], 
                       drawstyle = 'steps-mid', label = iden, alpha=0.6)
            ax[1].fill_between(spec['wave'][lyarange],
                               spec['spec'][lyarange] - spec['spec_err'][lyarange],
                               spec['spec'][lyarange] + spec['spec_err'][lyarange],
                               alpha=0.3, color='gray',
                               step='mid')
        ax[1].legend()
        plt.show()
        plt.close()

        # Get the hdul of the titular source spectrum
        titular_hdul = fits.open(spec_dir(titular_cluster) / filepattern(titular_source))
        # Update the spectrum and error in the hdul to be the new stacked spectrum and error
        if SPEC_SOURCE == 'APER':
            titular_hdul[1].data['spec'] = avgspec
            titular_hdul[1].data['spec_err'] = avgerr
        elif SPEC_SOURCE == 'R21':
            titular_hdul['DATA'].data = avgspec
            titular_hdul['STAT'].data = avgerr
        stacked_fname = filepattern(titular_source).replace(str(titular_source), stacked_iden)
        print(f"Saving stacked spectrum for {clus} source {titular_source} as {stacked_fname}...")
        titular_hdul.writeto(spec_dir(titular_cluster) / stacked_fname, overwrite=True)

In [ ]:
# check that the iden names aren't limited to 6 characters
print(f"Max length of iden names in megatab: {np.max([len(s) for s in megatab_reform['iden']])}")

In [ ]:
# Now any rows left that contain unique 'counters' that are there only because they were
# initially matched but later rejected due to contamination and/or poor SNR should have their
# COUNTER reset to zero

# Get unique values and their counts
unique_vals, indices, counts = np.unique(megatab_reform['COUNTER'].data, 
                                        return_index=True, return_counts=True)

# Find values that appear only once
unique_mask = counts == 1
unique_values_to_zero = unique_vals[unique_mask]

print(f"Setting {len(unique_values_to_zero)} unique COUNTER values to zero: {unique_values_to_zero}")

# Create a mask for rows with unique COUNTER values
mask = np.isin(megatab_reform['COUNTER'], unique_values_to_zero)

# Set those cells to 0
megatab_reform['COUNTER'][mask] = '0'

In [ ]:
# Now prepare to perform fitting of each of the new average spectra
wavedict = tgconst.wavedict

# First make a dictionary of lines
linedict = {}
for colname in megatab_reform.colnames:
    if 'SNR_' in colname:
        lname = colname.split('_')[-1]
        linedict[lname] = wavedict[lname]
        
print(f"We will be fitting the following lines:\n ")
for line in linedict.keys():
    print(f"{line} at {linedict[line]:.2f} A")

In [ ]:
def refit_stacked_spectrum(mega_table, clus, iden, show_all_plots=True):
    """
    Refit a stacked spectrum using initial guesses from the corresponding row of the provided mega table.
    In cases where no fit was previously obtained, revert to initial guesses from the R21 catalogues.
    
    Parameters
    ----------
        mega_table : astropy Table
            The megatable containing the fit results for all sources
        clus : str
            The cluster name
        iden : str
            The source identifier
        show_all_plots : bool, optional
            Whether to show all plots (default is True)
    
    Returns
    -------
        lya_fit_results : dict
            A dictionary containing the fit results for the Lyman alpha line
        other_fit_results : dict
            A dictionary containing the fit results for the other lines
    
    """
    # Get the row corresponding to the titular source -- this will give us best initial guesses for fitting
    row = mega_table[np.logical_and(mega_table['CLUSTER'] == clus, mega_table['iden'] == iden)]
    if len(row) != 1:
        raise ValueError(f"Could not uniquely identify {clus} {iden} in the provided table!")
    row = row[0]

    print("\n" + "=" * 80)
    print(f"Refitting stacked spectrum for {clus} {iden}...")

    # Load the stacked spectrum
    stacked_spec = tgio.load_spec(clus, iden, idfrom=row['idfrom'], spec_type=SPEC_TYPE, spec_source=SPEC_SOURCE)
    
    # Get wavelength, spectrum, and error arrays
    wave = stacked_spec['wave']
    spec = stacked_spec['spec']
    error = stacked_spec['spec_err']

    # Fit the Lyman alpha line first
    plot_dir = tgio.get_plot_dir(clus, iden)
    bootstrap_params = {
        'niter': 1000,
        'autocorrelation': False,
        'max_nfev': 2000,
        'errfunc': '84-16',
        'autocorrelation': False,
        'baseline_order': 1
    }
    lya_fit_results = lyafit.refit_lya_line(wave, spec, error, row, spec_type=SPEC_TYPE,
                                            plot_dir=plot_dir, bootstrap_params=bootstrap_params)
    
    param_dict = lya_fit_results.get('param_dict', None)

    fluxr = param_dict['FLUXR'] if param_dict is not None and 'FLUXR' in param_dict.keys() else None
    fluxr_err = lya_fit_results['error_dict']['FLUXR'] if param_dict is not None and 'FLUXR' in param_dict.keys() else None
    snrr = fluxr / fluxr_err if fluxr is not None and fluxr_err is not None and fluxr_err > 0 else 0.0
    print(f"Red peak SNR: {snrr:.2f}")

    if param_dict is not None:
        if 'FLUXB' in param_dict.keys():
            fluxb = param_dict['FLUXB']
            fluxb_err = lya_fit_results['error_dict']['FLUXB']
            snrb = fluxb / fluxb_err if fluxb_err > 0 else 0.0
            print(f"Blue peak SNR: {snrb:.2f}")

    # Now fit any other lines from the R21 catalogues
    hub_iden = iden.lstrip("S")
    if clus == "MACS0416S" and hub_iden in ["M2055", "M20355"]:
        hub_iden = hub_iden[:-2]
    line_table = tgcat.get_line_table(hub_iden, clus)
    # Remove any lines from the line table that are not in our linedict (i.e. that we aren't fitting)
    line_table = line_table[np.isin(line_table['LINE'], list(linedict.keys()))]
    if len(line_table) > 0:
        line_table = aptb.unique(line_table, keys='LINE') # Make sure there are no duplicate lines in the line table
    other_fit_results = {}

    # Pre-filter secondary doublet lines so the figure is sized correctly and has no blank panels
    filtered_lines = []
    for line_row in line_table:
        if np.any(tgconst.slines == line_row['LINE']):
            idx = np.where(tgconst.slines == line_row['LINE'])[0][0]
            primary = tgconst.flines[idx]
            if np.any(line_table['LINE'] == primary):
                print(f"Skipping secondary line {line_row['LINE']} as primary {primary} is also present.")
                continue
            else:
                print(f"Warning: Secondary line {line_row['LINE']} present without primary {primary}. Proceeding to fit.")
        filtered_lines.append(line_row)

    plot_lines = show_all_plots and len(filtered_lines) > 0

    fig, ax = None, None # Initialize fig and ax for plotting line fits if show_all_plots is True
    if plot_lines:
        # Make a grid of subplots sized to the number of lines actually being plotted
        n_lines = len(filtered_lines)
        n_cols = 5
        n_rows = (n_lines // n_cols) + (n_lines % n_cols > 0)
        fig, ax = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4), facecolor='white')
        ax = ax.flatten()

    ax_idx = 0 # Separate axis counter so unused axes are not left blank
    for line_row in filtered_lines:
        line_name = line_row['LINE']
        line_fit = tgfit.refit_other_line(wave, spec, error, row, line_row, width=25, 
                                    ax_in=ax[ax_idx] if plot_lines else None)
        if 'param_dict' in line_fit: # if the fit was successful, add it to the other_fit_results dictionary
            other_fit_results[line_name] = line_fit
        ax_idx += 1

    # Insert the new fit results back into the mega_table
    tgcat.insert_fit_results(mega_table, clus, iden, lya_fit_results, other_fit_results, replace_stacked=True)

    if plot_lines and fig is not None and ax is not None:
        # Remove any unused subplots
        for j in range(ax_idx, len(ax)):
            fig.delaxes(ax[j])
        plt.tight_layout()
        plt.savefig(plot_dir / f"{iden}_line_fits.png")
        plt.show()

    return lya_fit_results, other_fit_results


In [ ]:
# Make a copy of the megatab to work with
megatab_updated = copy.deepcopy(megatab_reform)

# Convert megatabs iden to 'object' type to allow longer strings
# Convert to numpy array first, then get string lengths
max_len = np.max([len(str(x)) for x in megatab_updated['iden']]) + 1
megatab_updated['iden'] = megatab_updated['iden'].astype(f'U{max_len}')

writename = f'./megatables/fit_catalog_qc_cpts_stack_{SPEC_TYPE}.fits' # Name for updated megatab

# Now loop over the stacked spectra and refit them
for row in megatab_reform[5:]:
    iden = row['iden']
    if iden[0] != 'S': # Only refit the stacked spectra, which have an S in front of their identifier
        continue
    # Figure out the original cluster name (required for MACS0416 as we have multiple subclusters)
    original_clusname = row['CLUSTER']
    
    refit_stacked_spectrum(megatab_updated, original_clusname, iden, show_all_plots=True)
    
    # Save the megatab after each fit in case something goes wrong
    megatab_updated.write(writename, overwrite=True)

In [ ]:
# This part gets rid of the counterpart rows and replaces with a single row
megatab_reform_nocounters = copy.deepcopy(megatab_updated)
counter_mask = megatab_updated['COUNTER'].data != '0'
counter_mask &= np.array([not s.startswith('S') for s in megatab_updated['iden'].data])
megatab_reform_nocounters = megatab_reform_nocounters[~counter_mask]

# Block to strip off additonal S if any are there
for i, iden in enumerate(megatab_reform_nocounters['iden']):
    if iden.startswith('SS'):
        megatab_reform_nocounters['iden'][i] = iden[1:]

megatab_reform_nocounters.write(writename, overwrite=True)